# ResStock 2024.2 Quick Start: Pull & EDA

A hands-on walkthrough for exploring **metadata** and **hourly load curves** from the
ResStock 2024.2 (AMY2018) release.

**What you'll do:**
1. Point at the right files for a state
2. Load and explore metadata (building characteristics)
3. Pull hourly loads for a small sample of buildings
4. Run basic EDA on loads and join back to metadata

**Prerequisites:** `just install` in the repo root; AWS credentials if reading from S3 (`just aws`).

**Further reading:** `context/docs/resstock_2024.2.md`, `context/code/data/parquet_reads_local_vs_s3.md`

## 0. Configuration

Pick a state and upgrade, then choose where to read from.

| Setting | Meaning |
|---------|--------|
| `STATE` | Two-letter code, e.g. `"NY"` or `"RI"` |
| `UPGRADE` | `"00"` = baseline stock, `"02"` = heat-pump measure package |
| `USE_LOCAL` | `True` on EC2/devcontainer (fast); `False` for S3 |
| `USE_SB` | `True` for Switchbox-enriched metadata (`metadata-sb.parquet`) |

Each modeled building represents **252.3 real dwellings** (ResStock sample weight). Use the
`weight` column when scaling up to population-level totals.

In [1]:
import sys
from pathlib import Path

# rate-design-platform modules (`data/`, `utils/loads.py`) live in a sibling repo.
# buildstock-fetch also has its own `utils/` package, so RDP must come first on sys.path.
RDP_ROOT = Path("/ebs/home/lily_switch_box/rate-design-platform")
if str(RDP_ROOT) not in sys.path:
    sys.path.insert(0, str(RDP_ROOT))

In [2]:
from pathlib import Path

import polars as pl
from data.eia.hourly_loads.eia_region_config import get_aws_storage_options
from utils.loads import ELECTRIC_LOAD_COL

# ── Edit these ──────────────────────────────────────────────────────────────
STATE = "NY"
UPGRADE = "00"  # try "02" later for the heat-pump upgrade
USE_LOCAL = True  # False → read from s3://data.sb/...
USE_SB = True  # False → raw NREL metadata.parquet
N_SAMPLE = 200  # buildings to pull hourly loads for
SEED = 42
# ────────────────────────────────────────────────────────────────────────────

RELEASE = "res_2024_amy2018_2"
RELEASE_SB = f"{RELEASE}_sb"
PATH_LOCAL = Path("/ebs/data/nrel/resstock")
PATH_S3 = "s3://data.sb/nrel/resstock"

release_name = RELEASE_SB if USE_SB else RELEASE
resstock_base = PATH_LOCAL / release_name if USE_LOCAL else f"{PATH_S3}/{release_name}"
storage_options = get_aws_storage_options() if not USE_LOCAL else None
meta_filename = "metadata-sb.parquet" if USE_SB else "metadata.parquet"

path_metadata = f"{resstock_base}/metadata/state={STATE}/upgrade={UPGRADE}/{meta_filename}"
path_loads_dir = f"{resstock_base}/load_curve_hourly/state={STATE}/upgrade={UPGRADE}"

print(f"Metadata:  {path_metadata}")
print(f"Loads dir: {path_loads_dir}")

Metadata:  /ebs/data/nrel/resstock/res_2024_amy2018_2_sb/metadata/state=NY/upgrade=00/metadata-sb.parquet
Loads dir: /ebs/data/nrel/resstock/res_2024_amy2018_2_sb/load_curve_hourly/state=NY/upgrade=00


## 1. Pull metadata

Metadata is **one parquet file per (state, upgrade)** — small enough to load entirely
(~30k rows × ~180 columns for NY). Always start here.

In [3]:
kwargs = {"storage_options": storage_options} if storage_options else {}
metadata = pl.read_parquet(path_metadata, **kwargs)

print(f"{metadata.height:,} buildings × {metadata.width} columns")
metadata.head(3)

33,790 buildings × 192 columns


upgrade,weight,in.sqft,in.representative_income,in.ahs_region,in.aiannh_area,in.area_median_income,in.ashrae_iecc_climate_zone_2004,in.ashrae_iecc_climate_zone_2004_2_a_split,in.bathroom_spot_vent_hour,in.battery,in.bedrooms,in.building_america_climate_zone,in.cec_climate_zone,in.ceiling_fan,in.census_division,in.census_division_recs,in.census_region,in.city,in.clothes_dryer,in.clothes_dryer_usage_level,in.clothes_washer,in.clothes_washer_presence,in.clothes_washer_usage_level,in.cooking_range,in.cooking_range_usage_level,in.cooling_setpoint,in.cooling_setpoint_has_offset,in.cooling_setpoint_offset_magnitude,in.cooling_setpoint_offset_period,in.corridor,in.county,in.county_and_puma,in.county_name,in.dehumidifier,in.dishwasher,in.dishwasher_usage_level,…,in.usage_level,in.utility_bill_electricity_fixed_charges,in.utility_bill_electricity_marginal_rates,in.utility_bill_fuel_oil_fixed_charges,in.utility_bill_fuel_oil_marginal_rates,in.utility_bill_natural_gas_fixed_charges,in.utility_bill_natural_gas_marginal_rates,in.utility_bill_propane_fixed_charges,in.utility_bill_propane_marginal_rates,in.utility_bill_scenario_names,in.utility_bill_simple_filepaths,in.vacancy_status,in.vintage,in.vintage_acs,in.water_heater_efficiency,in.water_heater_fuel,in.water_heater_in_unit,in.water_heater_location,in.weather_file_city,in.weather_file_latitude,in.weather_file_longitude,in.window_areas,in.windows,bldg_id,postprocess_group.has_hp,heats_with_electricity,heats_with_natgas,heats_with_oil,heats_with_propane,has_natgas_connection,mf_non_hvac_electricity_adjusted,has_child_under_6,has_person_over_60,has_disabled_person,is_vulnerable,postprocess_group.heating_type,postprocess_group.heating_type_v2
i64,f64,i64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,i64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,i64,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,str,str
0,252.301639,2179,89096.0,"""CBSA New York-Newark-Jersey Ci…","""No""","""80-100%""","""4A""","""4A""","""Hour17""","""None""","""5""","""Mixed-Humid""","""None""","""Standard Efficiency""","""Middle Atlantic""","""Middle Atlantic""","""Northeast""","""In another census Place""","""Electric""","""120% Usage""","""Standard""","""Yes""","""120% Usage""","""Electric Resistance""","""120% Usage""","""80F""","""Yes""","""5F""","""Night Setup""","""Not Applicable""","""G3601030""","""G3601030, G36003302""","""Suffolk County""","""None""","""318 Rated kWh""","""120% Usage""",…,"""High""",10,0.193439,"""0""","""3.211884615""","""11.25""","""1.196774216""","""0""","""3.052""","""Utility Rates - Fixed + Variab…","""data/simple_rates/State.tsv""","""Occupied""","""1960s""","""1960-79""","""FIXME Fuel Oil Indirect""","""Fuel Oil""","""Yes""","""Garage""","""Long Island Mac Art""",40.79,-73.1,"""F9 B9 L9 R9""","""Single, Clear, Non-metal, Exte…",70571,false,false,false,true,false,false,false,false,true,true,true,"""fossil_fuel""","""delivered_fuels"""
0,252.301639,1228,68718.0,"""Non-CBSA Middle Atlantic""","""No""","""100-120%""","""5A""","""5A""","""Hour19""","""None""","""3""","""Cold""","""None""","""Standard Efficiency""","""Middle Atlantic""","""Middle Atlantic""","""Northeast""","""In another census Place""","""Electric""","""100% Usage""","""EnergyStar""","""Yes""","""100% Usage""","""Gas""","""100% Usage""","""70F""","""No""","""0F""","""None""","""Not Applicable""","""G3600110""","""G3600110, G36000704""","""Cayuga County""","""None""","""None""","""100% Usage""",…,"""Medium""",10,0.193439,"""0""","""3.211884615""","""11.25""","""1.196774216""","""0""","""3.052""","""Utility Rates - Fixed + Variab…","""data/simple_rates/State.tsv""","""Occupied""","""<1940""","""<1940""","""Natural Gas Standard""","""Natural Gas""","""Yes""","""Unheated Basement""","""Oswego Co""",43.35,-76.39,"""F18 B18 L18 R18""","""Single, Clear, Non-metal""",71681,false,false,false

## 2. Metadata EDA

Column names follow ResStock conventions:
- `in.*` — input characteristics (geometry, HVAC, income, location)
- `out.*` — annual simulation outputs (in metadata for convenience)
- `postprocess_group.*` — Switchbox columns (only in `_sb` release)

In [4]:
# Schema peek — lots of columns; filter by prefix when hunting
in_cols = [c for c in metadata.columns if c.startswith("in.")]
out_cols = [c for c in metadata.columns if c.startswith("out.")]
sb_cols = [c for c in metadata.columns if c.startswith("postprocess_group.")]

print(f"in.* columns:  {len(in_cols)}")
print(f"out.* columns: {len(out_cols)}")
print(f"Switchbox:     {sb_cols}")

in.* columns:  176
out.* columns: 0
Switchbox:     ['postprocess_group.has_hp', 'postprocess_group.heating_type', 'postprocess_group.heating_type_v2']


In [5]:
# Heating fuel mix (primary heating system)
(
    metadata.group_by("in.hvac_heating_type_and_fuel")
    .agg(
        pl.len().alias("n_buildings"),
        pl.col("weight").sum().alias("total_weight"),
    )
    .sort("n_buildings", descending=True)
)

in.hvac_heating_type_and_fuel,n_buildings,total_weight
str,u32,f64
"""Natural Gas Fuel Furnace""",8567,2.1615e6
"""Natural Gas Shared Heating""",5978,1.5083e6
"""Fuel Oil Fuel Furnace""",3140,792227.145613
"""Natural Gas Fuel Wall/Floor Fu…",2953,745046.73917
"""Natural Gas Fuel Boiler""",2703,681971.329488
…,…,…
"""Other Fuel Shared Heating""",133,33556.117951
"""Other Fuel Fuel Furnace""",110,27753.18026
"""Propane Fuel Boiler""",90,22707.147486


In [6]:
# Building type and size
(
    metadata.group_by("in.geometry_building_type_recs")
    .agg(
        pl.len().alias("n"),
        pl.col("in.sqft").median().alias("median_sqft"),
    )
    .sort("n", descending=True)
)

in.geometry_building_type_recs,n,median_sqft
str,u32,f64
"""Single-Family Detached""",14111,1698.0
"""Multi-Family with 5+ Units""",11335,854.0
"""Multi-Family with 2 - 4 Units""",5857,854.0
"""Single-Family Attached""",1697,1678.0
"""Mobile Home""",790,1228.0


In [7]:
# Geography — county + PUMA is the smallest ResStock geography
(metadata.group_by("in.county_and_puma").len().sort("len", descending=True).head(10))

in.county_and_puma,len
str,u32
"""G3600610, G36003805""",564
"""G3600610, G36003806""",485
"""G3600610, G36003807""",432
"""G3601030, G36003305""",408
"""G3600610, G36003808""",397
"""G3600810, G36004103""",394
"""G3600610, G36003810""",381
"""G3600010, G36002002""",359
"""G3601190, G36003106""",338


In [8]:
# Switchbox heating classification (only in _sb metadata)
if "postprocess_group.heating_type" in metadata.columns:
    display(
        metadata.group_by("postprocess_group.heating_type")
        .agg(pl.len().alias("n"), pl.col("weight").sum().alias("weight"))
        .sort("n", descending=True)
    )
else:
    print("Set USE_SB = True to see Switchbox heating_type column")

postprocess_group.heating_type,n,weight
str,u32,f64
"""fossil_fuel""",29554,7.4565e6
"""electrical_resistance""",3070,774566.030902
"""heat_pump""",918,231612.904354
null,248,62570.806405


## 3. Sample buildings & pull hourly loads

Hourly loads are **one parquet per building** (`{bldg_id}-{upgrade}.parquet`, 8,760 rows).

> **S3 tip:** Don't `scan_parquet` an entire state from S3 — Polars probes every file (~30 min for NY).
> Build explicit paths for the buildings you need (shown below). On local EBS, scanning a partition is fine.

In [9]:
sample_ids = metadata.select("bldg_id").sample(min(N_SAMPLE, metadata.height), seed=SEED)["bldg_id"].to_list()
print(f"Sampled {len(sample_ids)} buildings: {sample_ids[:5]} ...")

Sampled 200 buildings: [209175, 137204, 393306, 473550, 220005] ...


In [10]:
# Build explicit file paths (works well on S3 and local)
upgrade_int = str(int(UPGRADE))  # "00" → "0"
load_paths = [f"{path_loads_dir}/{bldg_id}-{upgrade_int}.parquet" for bldg_id in sample_ids]

loads = pl.scan_parquet(load_paths, **kwargs).collect()
print(f"{loads.height:,} hourly rows for {loads['bldg_id'].n_unique()} buildings")
loads.select("bldg_id", "timestamp", ELECTRIC_LOAD_COL).head(5)

1,752,000 hourly rows for 200 buildings


bldg_id,timestamp,out.electricity.total.energy_consumption
i64,datetime[μs],f64
209175,2018-01-01 00:00:00,0.5
209175,2018-01-01 01:00:00,0.5
209175,2018-01-01 02:00:00,0.496
209175,2018-01-01 03:00:00,0.496
209175,2018-01-01 04:00:00,0.492


## 4. Load EDA

We use `out.electricity.total.energy_consumption` for electric kWh (matches what CAIRO bills).
Values are **kWh per hour**.

In [11]:
# Annual kWh per building
annual = (
    loads.group_by("bldg_id")
    .agg(pl.col(ELECTRIC_LOAD_COL).sum().alias("annual_kwh"))
    .sort("annual_kwh", descending=True)
)
annual.describe()

statistic,bldg_id,annual_kwh
str,f64,f64
"""count""",200.0,200.0
"""null_count""",0.0,0.0
"""mean""",291016.72,7197.248606
"""std""",157994.843876,6206.968081
"""min""",41.0,287.849903
"""25%""",170015.0,2699.231327
"""50%""",305087.0,5266.498832
"""75%""",429040.0,10146.414
"""max""",544638.0,42930.237


In [12]:
# Monthly load shape for one building (typical winter-peaking in NY)
one_bldg = sample_ids[0]
monthly = (
    loads.filter(pl.col("bldg_id") == one_bldg)
    .group_by("month")
    .agg(pl.col(ELECTRIC_LOAD_COL).sum().alias("monthly_kwh"))
    .sort("month")
)
print(f"Building {one_bldg}")
monthly

Building 209175


month,monthly_kwh
i8,f64
1,643.038
2,590.973
3,658.871
4,588.867
5,636.646
…,…
8,808.091
9,656.397
10,586.633


In [13]:
# Peak hour-of-day across the sample (unweighted)
(loads.group_by("hour").agg(pl.col(ELECTRIC_LOAD_COL).mean().alias("mean_kwh")).sort("hour"))

hour,mean_kwh
i8,f64
0,0.58446
1,0.56133
2,0.554406
3,0.551935
4,0.583152
…,…
19,1.080451
20,1.033337
21,0.884386


## 5. Join metadata + loads

Connect building characteristics to annual consumption. Use `weight` for
population-representative totals.

In [14]:
meta_cols = [
    c
    for c in (
        "bldg_id",
        "weight",
        "in.hvac_heating_type_and_fuel",
        "in.geometry_building_type_recs",
        "in.sqft",
        "postprocess_group.heating_type",
    )
    if c in metadata.columns
]

joined = metadata.select(meta_cols).join(annual, on="bldg_id", how="inner")
joined.head(5)

bldg_id,weight,in.hvac_heating_type_and_fuel,in.geometry_building_type_recs,in.sqft,postprocess_group.heating_type,annual_kwh
i64,f64,str,str,i64,str,f64
82978,252.301639,"""Natural Gas Fuel Furnace""","""Single-Family Detached""",1228,"""fossil_fuel""",4177.52
83932,252.301639,"""Natural Gas Shared Heating""","""Multi-Family with 2 - 4 Units""",1138,"""fossil_fuel""",4144.295242
108621,252.301639,"""Other Fuel Fuel Wall/Floor Fur…","""Single-Family Detached""",3310,"""fossil_fuel""",10203.089
131678,252.301639,"""Other Fuel Fuel Wall/Floor Fur…","""Mobile Home""",881,"""fossil_fuel""",10146.414
115429,252.301639,"""Natural Gas Shared Heating""","""Multi-Family with 5+ Units""",1138,"""fossil_fuel""",3622.606878


In [15]:
# Weighted mean annual kWh by heating type
(
    joined.group_by("in.hvac_heating_type_and_fuel")
    .agg(
        pl.len().alias("n"),
        (pl.col("annual_kwh") * pl.col("weight")).sum().alias("weighted_kwh"),
        pl.col("weight").sum().alias("total_weight"),
    )
    .with_columns((pl.col("weighted_kwh") / pl.col("total_weight")).alias("wmean_annual_kwh"))
    .sort("n", descending=True)
)

in.hvac_heating_type_and_fuel,n,weighted_kwh,total_weight,wmean_annual_kwh
str,u32,f64,f64,f64
"""Natural Gas Fuel Furnace""",45,8.2031e7,11353.573743,7225.116465
"""Natural Gas Shared Heating""",31,2.2213e7,7821.350801,2840.063202
"""Fuel Oil Fuel Furnace""",22,6.2736e7,5550.636052,11302.549097
"""Natural Gas Fuel Boiler""",20,2.4456e7,5046.032775,4846.611938
"""Natural Gas Fuel Wall/Floor Fu…",18,1.8878e7,4541.429497,4156.896563
…,…,…,…,…
"""Propane Fuel Wall/Floor Furnac…",1,1.6456e6,252.301639,6522.376
"""Other Fuel Shared Heating""",1,1.4953e6,252.301639,5926.685047
"""None""",1,264758.139425,252.301639,1049.371462


## 6. (Optional) Compare baseline vs heat-pump upgrade

Pull the same buildings under upgrade `02` to see how the HP measure package changes loads.
Re-run sections 3–5 with `UPGRADE = "02"`, or use the snippet below for a side-by-side on the sample.

In [16]:
HP_UPGRADE = "02"
path_loads_hp = f"{resstock_base}/load_curve_hourly/state={STATE}/upgrade={HP_UPGRADE}"
hp_paths = [f"{path_loads_hp}/{bldg_id}-{int(HP_UPGRADE)}.parquet" for bldg_id in sample_ids]

loads_hp = pl.scan_parquet(hp_paths, **kwargs).collect()
annual_hp = loads_hp.group_by("bldg_id").agg(pl.col(ELECTRIC_LOAD_COL).sum().alias("annual_kwh_hp"))

compare = (
    annual.join(annual_hp, on="bldg_id")
    .with_columns(
        (pl.col("annual_kwh_hp") - pl.col("annual_kwh")).alias("delta_kwh"),
        ((pl.col("annual_kwh_hp") / pl.col("annual_kwh") - 1) * 100).alias("pct_change"),
    )
    .join(metadata.select("bldg_id", "in.hvac_heating_type_and_fuel"), on="bldg_id")
)

print("Annual kWh change: baseline (00) → heat pump (02)")
compare.group_by("in.hvac_heating_type_and_fuel").agg(
    pl.col("delta_kwh").median().alias("median_delta_kwh"),
    pl.col("pct_change").median().alias("median_pct_change"),
).sort("median_delta_kwh", descending=True)

Annual kWh change: baseline (00) → heat pump (02)


in.hvac_heating_type_and_fuel,median_delta_kwh,median_pct_change
str,f64,f64
"""Propane Fuel Wall/Floor Furnac…",18585.25,284.946007
"""Propane Fuel Boiler""",11888.91,93.346318
"""Propane Fuel Furnace""",10536.789,108.74765
"""Fuel Oil Fuel Boiler""",6741.7145,79.822863
"""Natural Gas Fuel Wall/Floor Fu…",5663.6285,162.227765
…,…,…
"""Electricity MSHP""",-467.424772,-5.216118
"""Electricity ASHP""",-2500.754881,-24.794078
"""Electricity Baseboard""",-4286.278369,-40.898589


## Next steps

- **Utility assignment:** `metadata_utility/state={STATE}/utility_assignment.parquet` maps buildings to `sb.electric_utility`
- **Production helpers:** `utils/loads.py` (`scan_resstock_loads`, `ELECTRIC_LOAD_COL`)
- **S3 performance:** `context/code/data/parquet_reads_local_vs_s3.md`
- **Dataset docs:** `context/docs/resstock_2024.2.md` (measure packages, sample weights, geography limits)